# Generative AI Models and Probability

Generative artificial intelligence is built on a single mathematical idea: **learn a probability distribution over data, then sample from it**. Whether the model writes a sentence, paints an image, or synthesizes speech, it is estimating how likely different outputs are and choosing among them.

This notebook develops the probabilistic foundations that underpin every major generative architecture. We begin with the fundamental distinction between discriminative and generative modeling, survey the main families of generative models, review probability distributions and density functions, derive the maximum likelihood training objective, and connect these ideas to the neural networks used in practice.

The concepts here bridge the Mathematics for AI notebooks (probability and statistics) with the generative systems explored throughout this section. A clear grasp of $P(y \mid x)$ versus $P(x)$, joint and conditional distributions, and likelihood-based training is essential for understanding how language models, diffusion models, and other generators actually work.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(7)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. Discriminative vs Generative Models

Machine learning models can be classified by **what probability distribution they learn**. This distinction determines what the model can do after training.

### 1.1 Discriminative Models: Learning $P(y \mid x)$

A **discriminative model** learns the **conditional probability** of a label or target $y$ given input features $x$:

$$P(y \mid x)$$

The model draws a decision boundary between classes without modeling how the inputs themselves are distributed. It answers: *given this input, what label is most likely?*

Examples:
- Spam classification: $P(\text{spam} \mid \text{email text})$
- Image classification: $P(\text{cat} \mid \text{pixel values})$
- Sentiment analysis: $P(\text{positive} \mid \text{review})$

Logistic regression, support vector machines, and BERT classifiers are discriminative. They optimize cross-entropy or margin-based losses that directly estimate $P(y \mid x)$.

### 1.2 Generative Models: Learning $P(x)$ or $P(x \mid c)$

A **generative model** learns the **distribution of the data itself**:

$$P(x) \quad \text{or} \quad P(x \mid c)$$

where $c$ is an optional **condition** such as a class label, text prompt, or partial sequence. Because the model captures how data is produced, it can **sample** new data points that resemble the training distribution.

Examples:
- Unconditional image generation: $P(\text{image})$
- Text-to-image: $P(\text{image} \mid \text{caption})$
- Language modeling: $P(\text{next token} \mid \text{context})$

Generative models answer: *what does plausible data look like, and can I create more of it?*

### 1.3 Connecting the Two Views

Discriminative and generative perspectives relate through the **product rule** of probability:

$$P(x, y) = P(y \mid x) \cdot P(x) = P(x \mid y) \cdot P(y)$$

A generative classifier models $P(x \mid y)$ for each class and applies **Bayes' rule** to classify:

$$P(y \mid x) = \frac{P(x \mid y) \cdot P(y)}{P(x)}$$

A discriminative classifier models $P(y \mid x)$ directly, skipping the modeling of $P(x)$. For pure synthesis tasks — generating text, images, or audio — the generative view is primary: we need $P(x)$ or $P(x \mid c)$, not labels.

### 1.4 Side-by-Side Comparison

| Aspect | Discriminative | Generative |
|--------|---------------|------------|
| **Learns** | $P(y \mid x)$ | $P(x)$ or $P(x \mid c)$ |
| **Output** | Label, score, or prediction | New data sample |
| **Typical tasks** | Classification, regression, ranking | Text, image, audio generation |
| **Training focus** | Decision boundary | Data distribution |
| **Examples** | Logistic regression, ResNet classifier | GPT, Stable Diffusion, VAE |

The plot below illustrates the difference on a one-dimensional feature space. The discriminative view models how class probability changes with $x$ (a sigmoid curve). The generative view models the density of data itself (a mixture of two Gaussians).

In [ ]:
# Discriminative vs generative: two views of the same 1-D feature space

x = np.linspace(-4, 4, 400)

# Discriminative: P(y=1|x) — probability of class 1 given x
p_y_given_x = 1 / (1 + np.exp(-2 * x))

# Generative: P(x) — mixture of two Gaussians (how data is distributed)
p_x = (
    0.55 * np.exp(-0.5 * ((x + 1.2) / 0.7) ** 2)
    + 0.45 * np.exp(-0.5 * ((x - 1.5) / 1.0) ** 2)
)
p_x = p_x / np.trapezoid(p_x, x)  # normalize to a proper density

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(x, p_y_given_x, color="tab:blue")
ax[0].set_title("Discriminative: $P(y=1 \\mid x)$")
ax[0].set_xlabel("Feature $x$")
ax[0].set_ylabel("Probability")
ax[0].set_ylim(0, 1.05)

ax[1].plot(x, p_x, color="tab:green")
ax[1].set_title("Generative: $P(x)$")
ax[1].set_xlabel("Feature $x$")
ax[1].set_ylabel("Density")

plt.suptitle("Same feature space, different modeling objectives", y=1.02)
plt.tight_layout()
plt.show()

---

## 2. Types of Generative AI Models

Generative models differ in **how** they represent and sample from $P(x)$. Each architecture makes different trade-offs between sample quality, training stability, inference speed, and theoretical tractability.

### 2.1 Autoregressive Models

Autoregressive models factorize the joint distribution as a product of conditionals:

$$P(x_{1:T}) = \prod_{t=1}^{T} P(x_t \mid x_{1:t-1})$$

Generation proceeds sequentially: predict the next element, append it, repeat. This is the dominant paradigm for **text** (GPT, Llama) and also used for images (PixelCNN) and audio. Training is straightforward — maximize the likelihood of each next step — but sequential sampling can be slow for high-dimensional data.

### 2.2 Generative Adversarial Networks (GANs)

GANs use **adversarial training**: a **generator** $G$ maps random noise $z$ to synthetic samples, while a **discriminator** $D$ tries to distinguish real data from fakes. The generator learns to fool the discriminator:

$$\min_G \max_D \; \mathbb{E}_{x \sim P_{\text{data}}} [\log D(x)] + \mathbb{E}_{z \sim P_z} [\log(1 - D(G(z)))]$$

GANs produce sharp, realistic images but can suffer from **training instability** and **mode collapse** (generating limited variety).

### 2.3 Variational Autoencoders (VAEs)

VAEs learn a **latent variable model** $P(x) = \int P(x \mid z) P(z) \, dz$ where $z$ is a low-dimensional latent code. An encoder approximates the posterior $P(z \mid x)$, and a decoder models $P(x \mid z)$. Training maximizes a lower bound on log-likelihood (the **ELBO**). VAEs provide smooth latent spaces and principled probabilistic training, though outputs can appear blurry.

### 2.4 Diffusion Models

Diffusion models define a **forward noising process** that gradually corrupts data into noise, then learn to **reverse** it. Generation starts from pure noise and iteratively denoises to produce a sample. They dominate modern image synthesis (Stable Diffusion, DALL-E 2) because they achieve high quality and diversity, with fine-grained control through text conditioning.

### 2.5 Flow-Based Models

Flow models apply a sequence of **invertible transformations** $f$ to a simple base distribution $P_z$:

$$P(x) = P_z(f^{-1}(x)) \left| \det \frac{\partial f^{-1}}{\partial x} \right|$$

Because the transformation is invertible, the model computes **exact likelihood** in closed form. This makes density estimation rigorous, but architectural constraints (invertibility, Jacobian computation) limit flexibility compared to GANs or diffusion models.

### 2.6 Comparison Table

| Family | Mechanism | Likelihood | Strengths | Limitations | Primary Use |
|--------|-----------|------------|-----------|-------------|-------------|
| **Autoregressive** | Sequential conditionals $P(x_t \mid x_{<t})$ | Tractable (factorized) | Strong for sequences; simple training | Slow for images/video | LLMs, code, speech |
| **GAN** | Generator vs discriminator | Implicit (no explicit $P(x)$) | Sharp, realistic samples | Unstable training, mode collapse | Faces, style transfer |
| **VAE** | Latent variable + encoder-decoder | Lower bound (ELBO) | Smooth latent space, principled | Blurry outputs | Representation learning |
| **Diffusion** | Iterative denoising | Variational bound | High quality, diverse, controllable | Slow inference (many steps) | Text-to-image |
| **Flow** | Invertible transforms | Exact | Precise density estimation | Restricted architectures | Audio, density estimation |

Modern systems often combine ideas: diffusion models use transformer backbones; multimodal systems pair autoregressive text heads with diffusion image decoders. The family label matters less than understanding the underlying sampling mechanism.

---

## 3. Probability Distributions

A **probability distribution** assigns probabilities to outcomes. Generative models are, at their core, learned approximations of distributions over high-dimensional data.

### 3.1 Discrete Distributions

For **discrete** random variables (finite or countably infinite outcomes), a **probability mass function (PMF)** assigns a probability to each value:

$$P(X = x) = p(x), \quad \sum_x p(x) = 1$$

Examples in generative AI:
- Next-token distribution over a vocabulary: $P(\text{token} = w_i \mid \text{context})$
- Categorical distribution over image classes in conditional generation
- Bernoulli trials (binary outcomes)

Language models output discrete distributions via **softmax** over the vocabulary at each generation step.

### 3.2 Continuous Distributions

For **continuous** random variables (real-valued outcomes), individual points have zero probability. Instead, we describe the distribution with a **probability density function (PDF)** $f(x)$ such that:

$$P(a \leq X \leq b) = \int_a^b f(x) \, dx$$

Examples in generative AI:
- Latent vectors in VAEs (often Gaussian: $z \sim \mathcal{N}(0, I)$)
- Pixel intensities in image models (modeled as continuous or discretized)
- Noise distributions in diffusion models

### 3.3 Joint Distributions

When multiple random variables interact, we describe their **joint distribution** $P(X, Y)$ — the probability (or density) of $X$ and $Y$ taking specific values simultaneously.

For a sequence of tokens $x_1, \ldots, x_T$:

$$P(x_1, x_2, \ldots, x_T)$$

is the joint probability of the entire sequence. Autoregressive models decompose this joint into a chain of conditionals using the **chain rule**:

$$P(x_1, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1})$$

For paired data such as images and captions, the joint $P(\text{image}, \text{caption})$ can be modeled directly or factorized as $P(\text{image} \mid \text{caption}) \cdot P(\text{caption})$.

### 3.4 Conditional Distributions

A **conditional distribution** $P(X \mid Y = y)$ describes the distribution of $X$ given that $Y$ takes a specific value $y$:

$$P(X \mid Y) = \frac{P(X, Y)}{P(Y)}$$

Conditional distributions are central to generative AI:
- $P(\text{next word} \mid \text{previous words})$ — language modeling
- $P(\text{image} \mid \text{text prompt})$ — text-to-image generation
- $P(x_t \mid x_{t-1})$ — Markov chains and diffusion reverse steps

Conditioning is how we **steer** generation. A prompt, class label, or partial sequence fixes part of the joint distribution, and the model samples the remainder.

---

## 4. Probability Density Functions

For continuous generative models, understanding PDFs is essential. A PDF tells us **how densely** probability is spread across the value space.

### 4.1 What Is a PDF?

A function $f(x)$ is a valid **probability density function** if:

1. **Non-negativity:** $f(x) \geq 0$ for all $x$
2. **Normalization:** $\int_{-\infty}^{\infty} f(x) \, dx = 1$

Unlike a PMF, $f(x)$ can exceed 1 — it is a **density**, not a probability. The probability of falling in an interval is the **area under the curve**:

$$P(a \leq X \leq b) = \int_a^b f(x) \, dx$$

### 4.2 The Gaussian (Normal) Distribution

The most important continuous distribution in generative modeling is the **Gaussian** (normal) distribution:

$$f(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

Parameters:
- $\mu$ — **mean** (center of the distribution)
- $\sigma^2$ — **variance** (spread; $\sigma$ is the standard deviation)

Gaussians appear everywhere: VAE latent priors, diffusion noise schedules, initialization schemes, and as building blocks in mixture models.

### 4.3 Mixture Models

Real data is often **multimodal** — it clusters in several regions. A **mixture distribution** combines multiple simple distributions:

$$f(x) = \sum_{k=1}^{K} \pi_k \, f_k(x)$$

where $\pi_k \geq 0$, $\sum_k \pi_k = 1$, and each $f_k$ is a component PDF (often Gaussian). Mixture models can represent complex, multi-peaked data distributions with simple components — a pattern echoed in modern deep generative models that combine simpler learned distributions.

### 4.4 Normalization in Practice

When we define a PDF analytically or learn one with a neural network, we must ensure it integrates to 1. Techniques include:

- **Explicit normalization** — divide by the integral (as in the plot below)
- **Softmax** — converts unnormalized scores (logits) to a valid discrete distribution
- **Change of variables** — flow models use the Jacobian determinant to maintain normalization through invertible transforms

Neural networks typically output **unnormalized log-probabilities (logits)**; softmax or other normalization layers convert these to valid distributions at inference time.

In [ ]:
# PDF visualization: Gaussian and Gaussian mixture

def gaussian_pdf(x, mu, sigma):
    """Normal PDF with mean mu and standard deviation sigma."""
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

x = np.linspace(-6, 6, 500)

# Single Gaussian: mu=0, sigma=1
pdf_single = gaussian_pdf(x, mu=0, sigma=1)

# Gaussian mixture: two components
weights = [0.6, 0.4]
mus = [-2.0, 2.5]
sigmas = [0.8, 1.2]
pdf_mixture = sum(w * gaussian_pdf(x, mu, sigma) for w, mu, sigma in zip(weights, mus, sigmas))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(x, pdf_single, color="tab:blue", linewidth=2)
ax[0].fill_between(x, pdf_single, alpha=0.2, color="tab:blue")
ax[0].set_title("Gaussian PDF: $\\mathcal{N}(0, 1)$")
ax[0].set_xlabel("$x$")
ax[0].set_ylabel("$f(x)$")
ax[0].annotate(f"Area = {np.trapezoid(pdf_single, x):.4f}", xy=(0.05, 0.95), xycoords="axes fraction")

ax[1].plot(x, pdf_mixture, color="tab:green", linewidth=2)
for w, mu, sigma in zip(weights, mus, sigmas):
    ax[1].plot(x, w * gaussian_pdf(x, mu, sigma), linestyle="--", alpha=0.6, label=f"$\\pi={w}$, $\\mu={mu}$")
ax[1].fill_between(x, pdf_mixture, alpha=0.2, color="tab:green")
ax[1].set_title("Gaussian Mixture PDF")
ax[1].set_xlabel("$x$")
ax[1].set_ylabel("$f(x)$")
ax[1].legend(fontsize=8)
ax[1].annotate(f"Area = {np.trapezoid(pdf_mixture, x):.4f}", xy=(0.05, 0.95), xycoords="axes fraction")

plt.suptitle("Probability density functions integrate to 1", y=1.02)
plt.tight_layout()
plt.show()

---

## 5. Maximum Likelihood Estimation

How do generative models **learn** a distribution from data? The dominant framework is **Maximum Likelihood Estimation (MLE)** — find model parameters that make the observed training data as probable as possible.

### 5.1 Likelihood

Given a dataset $\mathcal{D} = \{x^{(1)}, x^{(2)}, \ldots, x^{(N)}\}$ and a parametric model $P_\theta(x)$ with parameters $\theta$, the **likelihood** is the probability of the data under the model, assuming samples are independent:

$$\mathcal{L}(\theta) = P_\theta(\mathcal{D}) = \prod_{i=1}^{N} P_\theta(x^{(i)})$$

Likelihood measures how well the model explains the observed data. Higher likelihood means the model assigns higher probability to the training examples.

### 5.2 Log-Likelihood

Products of many small probabilities cause **numerical underflow**. Taking the logarithm converts products to sums and is monotonic (preserving the argmax):

$$\log \mathcal{L}(\theta) = \sum_{i=1}^{N} \log P_\theta(x^{(i)})$$

Maximizing log-likelihood is equivalent to maximizing likelihood. In practice, we often work with the **average** log-likelihood per sample, or its negative:

$$-\frac{1}{N} \sum_{i=1}^{N} \log P_\theta(x^{(i)})$$

This negative average log-likelihood is the **cross-entropy loss** used to train language models and many other generative systems.

### 5.3 The MLE Objective

**Maximum Likelihood Estimation** finds parameters that maximize the likelihood (or log-likelihood) of the training data:

$$\theta^* = \arg\max_\theta \; \log \mathcal{L}(\theta) = \arg\max_\theta \; \sum_{i=1}^{N} \log P_\theta(x^{(i)})$$

Equivalently:

$$\theta^* = \arg\min_\theta \; \left(-\sum_{i=1}^{N} \log P_\theta(x^{(i)})\right)$$

For a Gaussian model, MLE has closed-form solutions: the optimal mean is the sample mean, and the optimal variance is the sample variance. For neural generative models, we optimize this objective with gradient descent.

### 5.4 MLE in Generative Model Training

| Model family | MLE formulation |
|-------------|----------------|
| **Autoregressive** | $\sum_t \log P_\theta(x_t \mid x_{<t})$ — next-token prediction loss |
| **VAE** | ELBO (evidence lower bound on $\log P_\theta(x)$) |
| **Flow** | $\log P_\theta(x) = \log P_z(f_\theta^{-1}(x)) + \log \left| \det J \right|$ — exact log-likelihood |
| **Diffusion** | Variational bound on $\log P_\theta(x)$ via denoising score matching |
| **GAN** | Does not use explicit MLE; optimizes adversarial game (implicit distribution) |

Despite architectural differences, most generative training objectives are variations of "make the model assign high probability to real data." Language model training is the most direct example: each token prediction step maximizes $\log P_\theta(\text{token} \mid \text{context})$.

In [ ]:
# Simple MLE demo: fit Gaussian mean and variance to observed data

true_mu, true_sigma = 3.0, 1.5
data = np.random.normal(true_mu, true_sigma, size=200)

# MLE closed-form estimates for Gaussian parameters
mle_mu = data.mean()
mle_sigma = data.std(ddof=0)  # MLE uses N denominator, not N-1

x = np.linspace(data.min() - 2, data.max() + 2, 300)
pdf_true = gaussian_pdf(x, true_mu, true_sigma)
pdf_mle = gaussian_pdf(x, mle_mu, mle_sigma)

# Log-likelihood of the data under the MLE model
log_likelihood = np.sum(np.log(gaussian_pdf(data, mle_mu, mle_sigma) + 1e-12))

plt.figure(figsize=(8, 4))
plt.hist(data, bins=25, density=True, alpha=0.4, color="tab:gray", label="Observed data")
plt.plot(x, pdf_true, linestyle="--", color="tab:red", label=f"True: $\\mu={true_mu}$, $\\sigma={true_sigma}$")
plt.plot(x, pdf_mle, color="tab:blue", linewidth=2, label=f"MLE: $\\hat{{\\mu}}={mle_mu:.2f}$, $\\hat{{\\sigma}}={mle_sigma:.2f}$")
plt.xlabel("$x$")
plt.ylabel("Density")
plt.title("MLE: Fitting a Gaussian to Data")
plt.legend()
plt.show()

print(f"True parameters:     mu = {true_mu:.4f}, sigma = {true_sigma:.4f}")
print(f"MLE estimates:       mu = {mle_mu:.4f}, sigma = {mle_sigma:.4f}")
print(f"Log-likelihood:      {log_likelihood:.2f}")
print(f"Avg log-likelihood:  {log_likelihood / len(data):.4f} per sample")

---

## 6. Generative AI Networks

Modern generative models implement probability distributions with **neural networks**. The network architecture determines what kind of data the model can represent and how efficiently it can sample.

### 6.1 Feedforward and Convolutional Networks

**Multilayer perceptrons (MLPs)** map inputs to outputs through stacked linear layers and nonlinear activations. In generative models, MLPs appear as decoders in VAEs (mapping latent codes to data) and as components in flow models.

**Convolutional Neural Networks (CNNs)** exploit spatial structure in images. They are used in:
- GAN generators and discriminators (e.g., StyleGAN)
- Diffusion model U-Net backbones (predicting noise at each step)
- VAE encoders and decoders for image data

CNNs preserve locality and translation equivariance, making them natural for visual generation.

### 6.2 Recurrent Networks

**Recurrent Neural Networks (RNNs)** and **LSTMs** process sequences one element at a time, maintaining a hidden state that summarizes the past. They were the foundation of early neural language models and sequence-to-sequence generation.

RNNs model $P(x_t \mid x_{1:t-1})$ through recurrent hidden state updates. While largely superseded by transformers for text, RNNs remain relevant for low-latency sequential tasks and as conceptual building blocks for understanding autoregressive generation.

### 6.3 Transformer Networks

**Transformers** use **self-attention** to relate every position in a sequence to every other position in parallel. They are the dominant architecture for modern generative AI:

- **Decoder-only transformers** (GPT, Llama) — autoregressive text generation
- **Encoder-decoder transformers** (T5, BART) — conditional generation (translation, summarization)
- **Transformer backbones in diffusion** — U-Net attention blocks for image denoising

Transformers output a probability distribution at each position (via softmax over the vocabulary), directly implementing the autoregressive factorization of $P(x_{1:T})$.

### 6.4 Encoder-Decoder and Latent Architectures

Many generative models use a two-part structure:

```
  Data x  ──→  [Encoder]  ──→  Latent z  ──→  [Decoder]  ──→  Reconstructed x'
                     │                              │
              compress to                    generate from
              latent code                    latent sample
```

- **VAE:** Encoder maps $x$ to distribution over $z$; decoder maps $z$ to $P(x \mid z)$
- **GAN:** Generator maps noise $z$ directly to synthetic $x$ (no explicit encoder)
- **Seq2seq:** Encoder processes input sequence; decoder generates output sequence

The latent space provides a compressed, structured representation that makes sampling and interpolation tractable.

### 6.5 Architecture Summary

| Architecture | Role in generation | Key models |
|-------------|-------------------|------------|
| **MLP** | Latent-to-data decoding | VAE decoder, flow layers |
| **CNN** | Spatial feature extraction and synthesis | GAN, diffusion U-Net |
| **RNN/LSTM** | Sequential autoregressive modeling | Early LMs, WaveNet |
| **Transformer** | Parallel sequence modeling, attention | GPT, Llama, Stable Diffusion |
| **U-Net** | Multi-scale denoising with skip connections | DDPM, Stable Diffusion |

The choice of architecture follows the data structure: sequences favor transformers and RNNs; images favor CNNs and U-Nets; low-dimensional latents favor MLPs. State-of-the-art systems combine multiple architectures — for example, a transformer text encoder conditioning a convolutional diffusion decoder.

---

## 7. Summary

**Discriminative models** learn $P(y \mid x)$ to classify or predict; **generative models** learn $P(x)$ or $P(x \mid c)$ to model and synthesize data. The product rule connects both views through joint distributions.

Five major generative families — **autoregressive**, **GAN**, **VAE**, **diffusion**, and **flow** — differ in how they represent and sample from distributions, each with distinct trade-offs in quality, speed, and training stability.

**Probability distributions** may be discrete (PMF over tokens) or continuous (PDF over latent vectors). **Joint** and **conditional** distributions factorize the full generative problem; autoregressive models exploit the chain rule to decompose $P(x_{1:T})$ into manageable steps.

**PDFs** describe how probability density is spread over continuous values, with normalization ensuring total probability equals 1. **Maximum Likelihood Estimation** trains generative models by maximizing $\sum_i \log P_\theta(x^{(i)})$ — the same principle behind cross-entropy loss in language models and ELBO training in VAEs.

**Neural architectures** — MLPs, CNNs, RNNs, transformers, and U-Nets — implement these probability models at scale. The next notebooks in this section build on these foundations to explore tokens, context windows, model selection, and inference in deployed generative systems.